ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('CC_GENERAL.csv') 

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df = df.drop(columns=['CUST_ID'])

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df = df.fillna(df.mean())

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(15, 12), bins=30)
plt.tight_layout()
plt.show()

: 

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='BALANCE', y='PURCHASES', alpha=0.6)
plt.title('Scatter Plot: Balance vs Purchases')
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='BALANCE', y='CASH_ADVANCE', alpha=0.6)
plt.title('Scatter Plot: Balance vs Cash Advance')
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia_values, marker='o', linestyle='--')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.xticks(k_range)
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []
k_silhouette_range = range(2, 11)

for k in k_silhouette_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_silhouette_range, silhouette_scores, marker='o', color='orange', linestyle='--')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Scores for Optimal K')
plt.xticks(k_silhouette_range)
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_df = pd.DataFrame({
    'K': list(k_silhouette_range),
    'Silhouette Score': silhouette_scores
})
print(silhouette_df)

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
final_k = 4  
kmeans_final = KMeans(n_clusters=final_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = cluster_labels

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean()
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
df['Cluster'].value_counts()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df['Cluster'], palette='Set1', alpha=0.7)
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('Visualizing Customer Segments using PCA')
plt.legend(title='Cluster')
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

1. Why is this an unsupervised learning problem?  The dataset has no target labels. the goal is to find natural groupings, not predict a specific outcome.

2. Why did we remove the `CUST_ID` column? It is a unique identifier with no statistical value, keeping it would distort distance calculations.

3. Which columns had missing values? 'MINIMUM_PAYMENTS' and 'CREDIT_LIMIT.'

4. How did you handle the missing values? By replacing them with the mean value of their respective columns.

5. Why is scaling important before applying K-Means? K-Means relies on distance. Unscaled features with large ranges (like balance) would mathematically dominate small-range features (like frequency).

6. Which K value did you choose? Explain your answer using the elbow method and silhouette score. K = 4. It marks the clear elbow bend where inertia drops slow down and maintains a strong silhouette score.

7. Based on the cluster summary table, describe each customer segment in your own words. 
Cluster 0: Low spenders with low balances (Inactive/Conservative)
Cluster 1: High balance users who mostly take out cash (Cash Borrowers)
Cluster 2: High spenders with high limits and frequent purchases (VIPs)
Cluster 3: Active buyers who prefer installment plans (One-off/Installment Shoppers)

8. Which cluster may represent high-value customers? The cluster with the highest PURCHASES and CREDIT_LIMIT (VIPs).

9. Which cluster may represent customers who rely more on cash advance? the cluster with the highest CASH_ADVANCE mean.

10. How can a company use these clusters for marketing strategy? Target VIPs with premium perks, cash borrowers with financial balance plans, and inactive users with cashback incentives.